# LIBRARY IMPORTS

In [1]:
###### LIBRARY IMPORTS ######
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torchaudio
import json
import torch.nn as nn
import time
import matplotlib.pyplot as plt
import re
from pathlib import Path
from typing import Any, Dict, Iterable, Optional, Union
from SSM_model_V1 import SSMPhonemeModel as SSMPhonemeModelV1
from SSM_model_V2 import SSMPhonemeModel as SSMPhonemeModelV2
from SSM_model_V3 import SSMPhonemeModel as SSMPhonemeModelV3
from SSM_model_V4 import SSMPhonemeModel as SSMPhonemeModelV4
from SSM_model_V5 import SSMPhonemeModel as SSMPhonemeModelV5
from S2T_CNN_time import S2T_CNN_time_V1
from TIMITDataset import TIMITDataset
import random
from utils import (collate_fn_ctc, save_training_result, collate_fn_ctc_time_domain_padding, 
                   collate_fn_ctc_time_domain_features_during_training, compute_batch_per_from_log_probs, decode_batch_to_phonemes, log_message, 
                   load_phoneme_intervals, calculate_phoneme_mel_statistics, ctc_greedy_decode_ids, 
                   align_phoneme_sequences, check_timit_boundaries)
from functools import partial # for the new collate function
import yaml
from train_eval_utils import (apply_scheduler, calculate_PER_for_batch, calculate_and_update_epoch_loss, 
                              check_gradients_for_inf, create_checkpoint, evaluate_one_batch, 
                              log_batch_loss_and_PER, log_early_stopping, log_epoch_info, 
                              log_training_time, qualitative_predictions, restore_best_model, 
                              stop_early, train_one_batch, calculate_test_dataset_metrics,
                              append_test_results, save_test_results_to_csv, count_invalid_ctc_samples,
                              log_test_phoneme_predictions, initialize_test_predictions_log, print_cuda_memory)
from Decoder import CTCDecoder
from Upsampler import Upsampler
from SSM_TU.patches.scheduler import CosineAnnealingWarmupRestarts
!pip install torch-audiomentations
from Augments import augments_weak

#import torch_xla
#import torch_xla.core.xla_model as xm

seed = 42 # define the seed for pytorch repeatability

# CONFIGURE COLAB

In [2]:
###### DEFINE PATHS EITHER FOR GOOGLE COLAB OR LOCALLY ######
import sys
import os

colab = False # decides whether colab or local pc used

if colab:
    from google.colab import drive
    from google.colab import runtime

    project_root_dir = os.getcwd() # get the root directory of the project
    data_root_dir = os.path.join(project_root_dir, "TIMIT") # define the path to the TIMIT dataset in colab
    drive.mount('/content/drive', force_remount=True) # mount the main drive directory to colab
    sys.path.append('/content/drive/MyDrive/thesis') # to import the SSM_TU directory we need its parent directory
    from SSM_TU.src.model.sequence_layer import SequenceLayer # import the SSM Layer directory
    vocab_root_dir = os.path.join(project_root_dir, "drive", "MyDrive", "thesis")
    ssm_tu_root_dir = vocab_root_dir
    s2s_edge_root_dir = vocab_root_dir

    # unzip the data
    if not os.path.exists("/content/TIMIT"):  # or your dataset folder
        !unzip -q /content/drive/MyDrive/thesis/TIMIT.zip -d /content/
    else:
        print("TIMIT folder already exists in /content/TIMIT folder")

    base_results_dir = r"/content/drive/MyDrive/thesis/results"

    # check the content of the main directory
    print("TIMIT dataset contents")
    !ls /content/TIMIT
else:
    project_root_dir = os.getcwd() # get the root directory of the project
    vocab_root_dir = project_root_dir
    data_root_dir = project_root_dir
    base_results_dir = os.path.join(project_root_dir, "results")
    ssm_tu_root_dir = vocab_root_dir
    s2s_edge_root_dir = vocab_root_dir

vocabulary_dir = os.path.join(vocab_root_dir, "vocabulary")
train_data_csv = os.path.join(data_root_dir, "data", "train_data.csv")
test_data_csv = os.path.join(data_root_dir, "data", "test_data.csv")
s2s_edge_root_dir = os.path.join(s2s_edge_root_dir, "S2S_model_descriptions")
ssm_tu_root_dir = os.path.join(ssm_tu_root_dir, "SSM_TU")

print(f"Project root directory: {project_root_dir}")
print(f"Data root directory: {data_root_dir}")
print(f"Vocabulary directory: {vocabulary_dir}")
print(f"SSM_TU_root_dir: {ssm_tu_root_dir}")
print(f"S2S_model_descriptions: {s2s_edge_root_dir}")
print(f"train_data.csv: {train_data_csv}")
print(f"test_data.csv: {test_data_csv}")
!ls /content/TIMIT/TIMIT


Project root directory: /Users/atis/Desktop/MasterArbeit/Code
Data root directory: /Users/atis/Desktop/MasterArbeit/Code
Vocabulary directory: /Users/atis/Desktop/MasterArbeit/Code/vocabulary
SSM_TU_root_dir: /Users/atis/Desktop/MasterArbeit/Code/SSM_TU
S2S_model_descriptions: /Users/atis/Desktop/MasterArbeit/Code/S2S_model_descriptions
train_data.csv: /Users/atis/Desktop/MasterArbeit/Code/data/train_data.csv
test_data.csv: /Users/atis/Desktop/MasterArbeit/Code/data/test_data.csv
ls: /content/TIMIT/TIMIT: No such file or directory


# CUDA

In [3]:
####### DEVICE SPECIFICATIONS ######

# GPU info (only works if CUDA runtime is active)
!nvidia-smi

complex_use = True  # MPS cannot handle complex tensors

print("PyTorch version:", torch.__version__)

# --- GPU / MPS checks ---
mps_available = torch.backends.mps.is_available()
cuda_available = torch.cuda.is_available()

print("MPS available:", mps_available)
print("MPS built:", torch.backends.mps.is_built())
print("CUDA available:", cuda_available)
print("CUDA built:", torch.backends.cuda.is_built())

# --- TPU check ---
tpu_available = False

#try:
#    tpu_available = True
#    tpu_device = xm.xla_device()
#    print("TPU detected:", tpu_device)

#except ImportError:
#    print("torch_xla not available → TPU not available")


# --- DEVICE SELECTION ---
if cuda_available:
    print("CUDA MEMORY")
    print("-----------")

    print(torch.cuda.get_device_name(0))
    print(f"Total memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    device = torch.device("cuda")
    print("🚀 Using GPU (CUDA)")
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print("Setting seed for cuda")

    torch.backends.cudnn.deterministic = True
    print("Cuda is deterministic")
    torch.backends.cudnn.benchmark = False
    print("Cuda benchmark = Fakse -> uses fixed algorithm") 
    torch.use_deterministic_algorithms(False) # if True, ctc doesnt work
#elif tpu_available:
#    device = tpu_device
#    print("🚀 Using TPU")
elif mps_available and not complex_use:
    device = torch.device("mps")
    print("🚀 Using GPU (MPS)")

else:
    device = torch.device("cpu")
    print("⚠️ Using CPU")

print("Final device:", device)

zsh:1: command not found: nvidia-smi
PyTorch version: 2.11.0
MPS available: False
MPS built: False
CUDA available: False
CUDA built: False
⚠️ Using CPU
Final device: cpu


# VOCABULARY LOADING

In [4]:
####### GPU ALLOCATION DEBUGGING FUNCTIONS #######
import torch


def tensor_memory_mib(tensor: torch.Tensor) -> float:
    """Memory occupied by the tensor's elements in MiB."""
    return tensor.numel() * tensor.element_size() / (1024 ** 2)


def print_dataset_memory(dataset, name: str) -> None:
    """Inspect the waveform tensor stored in a TIMITDataset."""

    print(f"\n===== {name} DATASET =====")

    waveforms = dataset.waveforms

    print(f"Number of recordings: {len(dataset)}")
    print(f"Waveform shape: {tuple(waveforms.shape)}")
    print(f"Waveform dtype: {waveforms.dtype}")
    print(f"Waveform device: {waveforms.device}")

    print(
        f"Waveform tensor memory: "
        f"{tensor_memory_mib(waveforms):.2f} MiB"
    )

    lengths = dataset.orig_waveform_lengths

    total_valid_samples = int(lengths.sum().item())
    total_stored_samples = waveforms.numel()

    padding_samples = (
        total_stored_samples - total_valid_samples
    )

    print(f"Valid audio samples: {total_valid_samples:,}")
    print(f"Stored samples: {total_stored_samples:,}")
    print(f"Padding samples: {padding_samples:,}")

    print(
        f"Padding percentage: "
        f"{100 * padding_samples / total_stored_samples:.2f}%"
    )

    print(
        f"Useful audio memory: "
        f"{total_valid_samples * waveforms.element_size() / 1024**2:.2f} MiB"
    )

    print(
        f"Padding memory: "
        f"{padding_samples * waveforms.element_size() / 1024**2:.2f} MiB"
    )


def print_cuda_memory(stage: str) -> None:
    """Print CUDA memory usage and GPU capacity."""

    print(f"\n===== CUDA MEMORY: {stage} =====")

    if not torch.cuda.is_available():
        print("CUDA is not available.")
        return

    device_index = torch.cuda.current_device()

    torch.cuda.synchronize(device_index)

    properties = torch.cuda.get_device_properties(device_index)

    free_bytes, total_bytes = torch.cuda.mem_get_info(device_index)

    allocated = torch.cuda.memory_allocated(device_index)
    reserved = torch.cuda.memory_reserved(device_index)
    peak = torch.cuda.max_memory_allocated(device_index)

    mib = 1024 ** 2

    print(f"GPU: {properties.name}")
    print(f"Total GPU memory: {total_bytes / mib:.2f} MiB")
    print(f"Free GPU memory: {free_bytes / mib:.2f} MiB")
    print(f"Used GPU memory (all processes): {(total_bytes-free_bytes)/mib:.2f} MiB")

    print(f"PyTorch allocated: {allocated / mib:.2f} MiB")
    print(f"PyTorch reserved: {reserved / mib:.2f} MiB")
    print(f"PyTorch peak allocated: {peak / mib:.2f} MiB")

In [5]:
####### LOAD VOCABULARY #######

# Select phoneme vocabulary version.
# Allowed values: 39 or 61
phoneme_set_version = 61

# ---------------------------------------------------------
# Validate phoneme set
# ---------------------------------------------------------
if phoneme_set_version not in {39, 61}:
    raise ValueError(
        f"Unsupported phoneme_set_version={phoneme_set_version}. "
        "Expected either 39 or 61."
    )

# ---------------------------------------------------------
# Define vocabulary filenames
# ---------------------------------------------------------
phoneme_to_idx_vocab_filename = (
    f"phoneme_to_idx_{phoneme_set_version}.json"
)

idx_to_phoneme_vocab_filename = (
    f"idx_to_phoneme_{phoneme_set_version}.json"
)

phoneme_to_idx_vocab_path = os.path.join(
    vocabulary_dir,
    phoneme_to_idx_vocab_filename
)


idx_to_phoneme_vocab_path = os.path.join(
    vocabulary_dir,
    idx_to_phoneme_vocab_filename
)

# ---------------------------------------------------------
# Check if the vocabulary files exist
# ---------------------------------------------------------
if not os.path.exists(phoneme_to_idx_vocab_path):
    raise FileNotFoundError(
        f"Vocabulary file not found:\n{phoneme_to_idx_vocab_path}"
    )

if not os.path.exists(idx_to_phoneme_vocab_path):
    raise FileNotFoundError(
        f"Vocabulary file not found:\n{idx_to_phoneme_vocab_path}"
    )

# ---------------------------------------------------------
# Load phoneme -> index vocabulary
# ---------------------------------------------------------
with open(phoneme_to_idx_vocab_path, "r", encoding="utf-8") as file:
    phoneme_to_idx_vocab = json.load(file)

# ---------------------------------------------------------
# Load index -> phoneme vocabulary
# ---------------------------------------------------------
with open(idx_to_phoneme_vocab_path, "r", encoding="utf-8") as file:
    idx_to_phoneme_vocab = json.load(file)

# JSON stores dictionary keys as strings, therefore convert them back to integers.
idx_to_phoneme_vocab = {
    int(idx): phoneme
    for idx, phoneme in idx_to_phoneme_vocab.items()
}

print(
    f"Successfully loaded the {phoneme_set_version}-phoneme vocabulary.\n"
    f"Vocabulary size (including CTC blank): {len(phoneme_to_idx_vocab)}"
)

Successfully loaded the 61-phoneme vocabulary.
Vocabulary size (including CTC blank): 62


# DEFINE MODEL/LOSS/OPTIMIZER

In [6]:
####### Model parameters checking function #######
def check_model_parameters(model):

    for name, param in model.named_parameters():

        if not torch.isfinite(param).all():

            print(
                f"NON-FINITE PARAMETER: {name} | "
                f"NaNs={torch.isnan(param).sum().item()} | "
                f"Infs={torch.isinf(param).sum().item()}"
            )

            raise RuntimeError(
                f"Model parameter became non-finite: {name}"
            )

In [8]:
####### Testing relevant objects ######
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

trained_model_dir_name = "S2S_Edge_large_30epoch_subsample1_2_8_32_128_512_time"
config_file_path = f"/Users/atis/Desktop/MasterArbeit/Code/S2S_Edge_trained_models/{trained_model_dir_name}/run_config.json"

with open(config_file_path, "r") as fopen:
    config_dict = json.load(fopen)
    
    d_out = config_dict["model_info"]["ssm_d_out"]
    d_state = config_dict["model_info"]["ssm_d_state"]
    d_in = config_dict["model_info"]["ssm_d_in"]
    ssm_num_layers = config_dict["model_info"]["num_of_ssm_layers"]  
    ssm_dropout = config_dict["model_info"]["ssm_dropout"]
    linear_dropout = config_dict["model_info"]["linear_dropout"]
    input_bias = config_dict["model_info"]["input_bias"]
    output_bias = config_dict["model_info"]["output_bias"]
    complex_output = config_dict["model_info"]["complex_output"]
    B_C_init = config_dict["model_info"]["B_C_init"]
    bias_init = config_dict["model_info"]["bias_init"]
    stability = config_dict["model_info"]["stability"]
    norm = config_dict["model_info"]["norm"]
    norm_type = config_dict["model_info"]["norm_type"]
    act = config_dict["model_info"]["act"]
    trainable_SkipLayer = config_dict["model_info"]["trainable_SkipLayer"]
    apply_rzoh = config_dict["model_info"]["apply_rzoh"]
    linear_dropout = config_dict["model_info"]["linear_dropout"]
    train_phoneme_to_idx_vocab = config_dict["phoneme_to_idx_vocab"]
    ssm_vocab_size = len(train_phoneme_to_idx_vocab)
    subsampling = config_dict["model_info"]["sub_sampling"]
    rzoh_probabilities = config_dict["model_info"]["rzoh_probabilities"] if apply_rzoh else None
    sub_sampling = config_dict["model_info"]["sub_sampling"]

    # mel transform parameters
    mel_transform_dict = config_dict["mel_transform_dict"]
    if mel_transform_dict:
        mel_transform_n_mels = mel_transform_dict["mel_transform_n_mels"]
        mel_transform_nfft = mel_transform_dict["mel_transform_nfft"]
        mel_transform_hop_length = mel_transform_dict["mel_transform_hop_length"]
        mel_transform_win_length = mel_transform_dict["mel_transform_win_length"]
        mel_transform_sample_rate = mel_transform_dict["mel_transform_sample_rate"]

########## MODEL DEFINITION ##########
print_cuda_memory("A. Before model creation")
ssm_model = SSMPhonemeModelV4(d_state=d_state,
                              d_in=d_in,
                              d_out=d_out,
                              num_of_ssm_layers=ssm_num_layers,
                              vocab_size=ssm_vocab_size,
                              ssm_dropout=ssm_dropout,
                              input_bias = input_bias,
                              output_bias = output_bias,
                              complex_output = complex_output,
                              B_C_init=B_C_init,
                              bias_init = bias_init,
                              stability=stability,
                              linear_dropout=linear_dropout,
                              norm=norm,
                              norm_type=norm_type,
                              act=act,
                              trainable_SkipLayer=trainable_SkipLayer,
                              rzoh_probabilities=rzoh_probabilities)

print_cuda_memory("B. After model creation")
ssm_model.to(device=device)
print_cuda_memory("C. After moving model to CUDA")

# transformation definition is later because of trainable_SkipLayer and d_in_first_layer definition
########## MEL TRANSFORMATION and DICTIOMARY DEFINITION ##########
if mel_transform_dict:
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=mel_transform_sample_rate,
        n_fft=mel_transform_nfft, # FFT window size => how many samples are used for frequency analysis
        hop_length=mel_transform_hop_length, # step size between windows => at 16kHz a hop length of 160 means a stepsize of 10ms
        win_length=mel_transform_win_length, # Actual window size applied before FFT => at 16kHz a window size of 400 means 25ms window length
        n_mels=mel_transform_n_mels # Number of frequency bins after Mel conversion => number of frequency elements in each window AFTER conversion
    )
    print(f"✅ Mel spectrogram transformation is used")
else: 
    mel_transform_dict = None
    print(f"✅ Mel spectrogram transformation is  NOT used")


print(f"✅ Input dimension: {d_in}")
print(f"✅ Output dimension: {d_out}")

# CTC loss function
ctc_zero_infinity = False
loss_fn = nn.CTCLoss(blank=0,
                     zero_infinity=ctc_zero_infinity) # to nullyify the infinity effects of possible very short sequences




===== CUDA MEMORY: A. Before model creation =====
CUDA is not available.

===== CUDA MEMORY: B. After model creation =====
CUDA is not available.

===== CUDA MEMORY: C. After moving model to CUDA =====
CUDA is not available.
✅ Mel spectrogram transformation is  NOT used
✅ Input dimension: [1, 16, 32, 64, 128, 256]
✅ Output dimension: [16, 32, 64, 128, 256, 512]


# DATASET AND DATALOADER

In [9]:
####### DATASETS AND DATALOADERS #######
print_cuda_memory("BEFORE DATASET CREATION")

padding_after_mel_transform = False

collate_function = ""

if padding_after_mel_transform:

    dataloader_collate_fn = collate_fn_ctc
    collate_function = "collate_fn_ctc"

else:

    if not mel_transform_dict:
        dataloader_collate_fn = (
            collate_fn_ctc_time_domain_features_during_training
        )

        collate_function = (
            "collate_fn_ctc_time_domain_features_during_training"
        )
        print(f"✅ Mel spectrogram transformation is NOT used")
    else:
        dataloader_collate_fn = partial(
            collate_fn_ctc_time_domain_padding,
            mel_transform=mel_transform,
            hop_length=mel_transform_hop_length,
            apply_rzoh=apply_rzoh
        )

        collate_function = "collate_fn_ctc_time_domain_padding"
        print(f"✅ Mel spectrogram transformation is used")


# ============================================================
# TRAINING DATASET
# ============================================================

batch_size = 8

train_dataset = TIMITDataset(
    csv_file=train_data_csv,
    train_or_test="TRAIN",
    root_dir=project_root_dir,
    phoneme_to_idx_vocab=phoneme_to_idx_vocab,
    transform=None,
)

print_dataset_memory(train_dataset, "TRAIN")
print_cuda_memory("AFTER TRAIN DATASET")


# ============================================================
# VALIDATION DATASET
# ============================================================

val_dataset = TIMITDataset(
    csv_file=test_data_csv,
    train_or_test="TEST",
    root_dir=project_root_dir,
    phoneme_to_idx_vocab=phoneme_to_idx_vocab,
    transform=None,
)

print_dataset_memory(val_dataset, "VALIDATION")
print_cuda_memory("AFTER VALIDATION DATASET")


# ============================================================
# DATALOADERS
# ============================================================

g = torch.Generator()
g.manual_seed(seed)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=dataloader_collate_fn,
    generator=g
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=dataloader_collate_fn
)

print_cuda_memory("AFTER DATALOADER CREATION")



===== CUDA MEMORY: BEFORE DATASET CREATION =====
CUDA is not available.
✅ Mel spectrogram transformation is NOT used
TRAIN Dataset creation process...
Filtering dataframe to only include TRAIN samples
Checked -> whole dataset belongs to TRAIN
Creating cache directory
Creating the samples dictionary...
Loading Cache dataset...
TRAIN dataset created 

===== TRAIN DATASET =====
Number of recordings: 4620
Waveform shape: (4620, 1, 124672)
Waveform dtype: torch.float32
Waveform device: cpu
Waveform tensor memory: 2197.21 MiB
Valid audio samples: 227,089,204
Stored samples: 575,984,640
Padding samples: 348,895,436
Padding percentage: 60.57%
Useful audio memory: 866.28 MiB
Padding memory: 1330.93 MiB

===== CUDA MEMORY: AFTER TRAIN DATASET =====
CUDA is not available.
TEST Dataset creation process...
Filtering dataframe to only include TEST samples
Checked -> whole dataset belongs to TEST
Creating cache directory
Creating the samples dictionary...
Loading Cache dataset...
TEST dataset create

# DECODER

In [10]:
####### DEFINE THE DECODER #######
blank_idx = 0
beam_width = 5
top_paths = 1
decoding_method = "beam_search" # Options: 1. greedy
                           #          2. beam_search
ctc_decoder = CTCDecoder(blank_idx=blank_idx,
                         beam_width=beam_width,
                         top_paths=top_paths,
                         method=decoding_method)

print(f"✅ Decoder beam width: {beam_width}")
print(f"✅ Decoder top paths: {top_paths}")
print(f"✅ Decoder decoding method: {decoding_method}")


✅ Decoder beam width: 5
✅ Decoder top paths: 1
✅ Decoder decoding method: beam_search


# TESTING CONFIGURATION

In [11]:
####### MODEL CONFIGURATION #######
# Path relative to /content/drive/MyDrive/thesis
# Example:
# path_to_weights = "/results/SSM_V5/experiment_01"
loaded_model_name = "S2S_Edge_large_30epoch_subsample1_2_8_32_128_512_time"
path_to_weights = f"S2S_Edge_trained_models/{loaded_model_name}"

phoneme_mapping_type = "61_to_39"
phoneme_mapping_path = f"{project_root_dir}/vocabulary/{phoneme_mapping_type}_vocab.json"

with open(phoneme_mapping_path, "r") as fopen:
    phoneme_mapping = json.load(fopen)

if colab:
    weights_path = f"/content/drive/MyDrive/thesis/results/{path_to_weights}/ssm_model_state_dict.pt"
    path_to_test_save = f"/content/drive/MyDrive/thesis/results/{path_to_weights}/test_results/{decoding_method}"
else:
    weights_path = f"{project_root_dir}/{path_to_weights}/ssm_model_state_dict.pt"
    path_to_test_save = f"{project_root_dir}/{path_to_weights}/test_results/{decoding_method}"

print("Loading weights from:")
print(weights_path)

###############################################################
# Check that the weight file exists
if not os.path.exists(weights_path):
    raise FileNotFoundError(
        f"Model weights could not be found:\n{weights_path}"
    )

###############################################################
# Load trained weights into the already instantiated model
state_dict = torch.load(
    weights_path,
    map_location=device,
    weights_only=True
)

# Create the test results directory
os.makedirs(path_to_test_save,
            exist_ok=True)

ssm_model.load_state_dict(state_dict)
print(f"path_to_test_save:\n{path_to_test_save}")


Loading weights from:
/Users/atis/Desktop/MasterArbeit/Code/S2S_Edge_trained_models/S2S_Edge_large_30epoch_subsample1_2_8_32_128_512_time/ssm_model_state_dict.pt
path_to_test_save:
/Users/atis/Desktop/MasterArbeit/Code/S2S_Edge_trained_models/S2S_Edge_large_30epoch_subsample1_2_8_32_128_512_time/test_results/beam_search


In [12]:
####### SUBSAMPLING CONFIGURATION #######
from utils import sweep
subsampling_type = "custom_list" # types are: sweep, custom_list, None

if subsampling_type == "sweep":
    max_scale = 8 # maximum factor of downsampling compared to the first initial layer

    step_scale_configs = sweep(
        current_scale=1,
        remaining_depth=len(ssm_model.ssm_layers),
        max_scale=max_scale
    )
    print(f"Generated {len(step_scale_configs)} configurations.")
elif subsampling_type == "custom_list":
    step_scale_configs = [sub_sampling]
else:
    step_scale_configs = [
        [1] * len(ssm_model.ssm_layers)
    ]

for config in step_scale_configs:
    print(config)

# Move model to selected device and switch to evaluation mode
ssm_model = ssm_model.to(device)

print("✅ Trained model weights loaded successfully.")

[1, 2, 8, 32, 128, 512]
✅ Trained model weights loaded successfully.


In [13]:
####### UPSAMPLING CONFIGURATION #######
use_upsampling = False
upsampling_method = "linear_pytorch"
upsampling_factor = 8

if use_upsampling:
    upsampler = Upsampler(method=upsampling_method)
else: 
    upsampling_method = None
    upsampler = None
    upsampling_factor = 0

# TESTING

In [15]:
####### TESTING #######

ssm_model.eval()
ssm_model.to(device)

# each entry is a dictionary with the results of the step_scale configurations generated by sweep
test_results = []

################### BASELINE COMPUTATIONAL COST ###################
# set the model to its original configuration
baseline_step_scales = [1] * len(ssm_model.ssm_layers)

enable_prediction_logging = False

# Calculate reference computational cost
default_macs = ssm_model.get_number_of_MACs()
print(path_to_test_save)
# Count model parameters
number_of_parameters = ssm_model.get_number_of_parameters()
number_of_trainable_parameters = ssm_model.get_number_of_trainable_parameters()


with torch.no_grad():
    for config_idx, step_scales in enumerate(step_scale_configs):

        print(
            f"\nTesting configuration "
            f"{config_idx + 1}/{len(step_scale_configs)}: {step_scales}"
        )

        # Apply the current subsampling configuration
        ssm_model.set_step_scale(step_scales)

        # calculate comuptational cost
        macs = ssm_model.get_number_of_MACs()
        macs_percent = ssm_model.get_MACs_percent(default_macs=default_macs,
                                                  macs=macs)

        # Initialize metrics for this configuration
        total_edit_distance = 0
        total_target_length = 0
        true_val_batch_loss = 0.0

        # when the subsampled input to ctc is shorter then the number of phonemes themselves
        invalid_ctc_samples_before_upsampling = 0
        invalid_ctc_samples_after_upsampling = 0 


        ########## INITIALIZE TEST PREDICTION LOG ##########
        if enable_prediction_logging:
            log_file_predictions, log_file_path, step_scales_string = initialize_test_predictions_log(path_to_test_save=path_to_test_save,
                                                                                                    step_scales=step_scales,
                                                                                                    decoding_method=decoding_method)

        recording_offset = 0

        # Iterate through the entire test dataset
        # X_val: torch.Tensor, [B,T_padded, F], input recordings padded to equal temporal length
        # X_val_length: torch.Tensor, [B], valid original input timesteps for each recording WITHOUT THE PADDING
        # y_val: torch.tensor [N], Ground truth PHONEME INDICES concatenated across the batch,
        # y_val_length: torch.Tensor, [B], number of GROUND-TRUTH phonemes in each recording, -> y_val_length[0] is the number of target phonemes of the first recording
        for batch_idx, (X_val, X_val_length, y_val, y_val_length) in enumerate(val_dataloader):
            print(f"Evaluate batch: {batch_idx}/{len(val_dataloader)}")
            ###### EVALUATE ONE BATCH ######
            # log_probs: torch.Tensor, [B, T_subsampled, vocabulary_size], predicted log probability of every phoneme class at every output time step for every recording in the batch
            # valid_output_lengths: torch.tensor, [B], the unpadded valid timesteps of each recordings AFTER THE SUBSAMPLING (if it was applied)
            log_probs, log_probs_ctc, valid_output_lengths, true_val_batch_loss, val_loss  = evaluate_one_batch(X_val=X_val,
                                                                                                                X_val_length=X_val_length,
                                                                                                                y_val=y_val,
                                                                                                                y_val_length=y_val_length,
                                                                                                                device=device,
                                                                                                                ssm_model=ssm_model,
                                                                                                                loss_fn=loss_fn,
                                                                                                                true_val_batch_loss=true_val_batch_loss,
                                                                                                                test_subsampling=True,
                                                                                                                upsampler=upsampler,
                                                                                                                upsampling_factor=upsampling_factor)

            # Track how many records have insufficient output time steps for CTC alignment
            # it can happen if subsampling is too string and the output lengths is shorter then the number of phonemes making up the sample
            if upsampler is not None:
                original_output_lengths = ssm_model.get_output_lengths(
                    X_val_length
                )

                invalid_ctc_samples_before_upsampling += count_invalid_ctc_samples(
                    targets=y_val,
                    target_lengths=y_val_length,
                    output_lengths=original_output_lengths
                )
            else:
                invalid_ctc_samples_before_upsampling = "No Upsampling was used"

            
            invalid_ctc_samples_after_upsampling += count_invalid_ctc_samples(targets=y_val,
                                                                              target_lengths=y_val_length,
                                                                              output_lengths=valid_output_lengths)


            ###### CALCULATE THE PER value for the current batch
            # log_probs: torch.Tensor - floating point, [B, T_subsampled, vocabulary_size], predicted log probability of every phoneme class at every output time step for every recording in the batch
            # valid_output_lengths: torch.tensor - integer, [B], the unpadded valid timesteps of each recordings AFTER THE SUBSAMPLING (if it was applied)
            # y_val: torch.Tensor - integer, [N], Ground truth PHONEME INDICES concatenated across the batch,
            # y_length: torch.Tensor - integer, [B], number of GROUND-TRUTH phonemes in each recording, -> y_val_length[0] is the number of target phonemes of the first recording
            # blank_idx: integer, 0, the index of the blank character in the vocabulary
            # total_edit_distance: int, initially 0, accumulated at every batch
            # total_target_length: int, initially 0, accumulated at every batch,
            # decoder: CTCDecoder, decoder object to use either ctc_greedy decoding or ctc_beam_search on the log_probs
            batch_val_per, total_edit_distance, total_target_length = calculate_PER_for_batch(log_probs=log_probs,
                                                                                              X_length=valid_output_lengths,
                                                                                              y=y_val,
                                                                                              y_length=y_val_length,
                                                                                              total_edit_distance=total_edit_distance,
                                                                                              total_target_length=total_target_length,
                                                                                              decoder=ctc_decoder,
                                                                                              use_vocab_conversion=True,
                                                                                              phoneme_mapping=phoneme_mapping)

            ########## LOG ALL PREDICTED PHONEMES ##########
            if enable_prediction_logging:
                recording_offset = log_test_phoneme_predictions(log_probs=log_probs,
                                                                valid_output_lengths=valid_output_lengths,
                                                                targets=y_val,
                                                                target_lengths=y_val_length,
                                                                decoder=ctc_decoder,
                                                                idx_to_phoneme_vocab=idx_to_phoneme_vocab,
                                                                log_file=log_file_predictions,
                                                                batch_idx=batch_idx,
                                                                recording_offset=recording_offset)

        ########## CLOSE PREDICTION LOG FILE ##########
        if enable_prediction_logging:
            log_file_predictions.close()
            print(f"Saved phoneme predictions to: {log_file_path}")

        ########## CALCULATE DATASET LOSS AND PER (LOSS AND PER ACROSS THE WHOLE TEST DATASET)##########
        dataset_loss, dataset_per = calculate_test_dataset_metrics(true_val_batch_loss=true_val_batch_loss,
                                                                   val_dataloader=val_dataloader,
                                                                   total_edit_distance=total_edit_distance,
                                                                   total_target_length=total_target_length)

        ########## APPEND TEST RESULTS OF THE CURRENT STEP SCALE CONFIG ##########
        append_test_results(test_results=test_results,
                            step_scales=step_scales,
                            dataset_loss=dataset_loss,
                            dataset_per=dataset_per,
                            total_edit_distance=total_edit_distance,
                            total_target_length=total_target_length,
                            upsampling_method=upsampling_method,
                            upsampling_factor=upsampling_factor,
                            invalid_ctc_samples_before_upsampling=invalid_ctc_samples_before_upsampling,
                            invalid_ctc_samples_after_upsampling=invalid_ctc_samples_after_upsampling,
                            macs=macs,
                            macs_percent=macs_percent,
                            params=number_of_parameters,
                            trainable_params=number_of_trainable_parameters,
                            decoder=decoding_method)

    ########## APPEND TEST RESULTS OF THE CURRENT STEP SCALE CONFIG ##########
    save_test_results_to_csv(test_results=test_results,
                             test_save_path=path_to_test_save,
                             upsampling_factor=upsampling_factor)
    print(f"Results saved into {path_to_test_save}")
        

/Users/atis/Desktop/MasterArbeit/Code/S2S_Edge_trained_models/S2S_Edge_large_30epoch_subsample1_2_8_32_128_512_time/test_results/beam_search

Testing configuration 1/1: [1, 2, 8, 32, 128, 512]
Evaluate batch: 0/210
Evaluate batch: 1/210
Evaluate batch: 2/210
Evaluate batch: 3/210
Evaluate batch: 4/210
Evaluate batch: 5/210
Evaluate batch: 6/210
Evaluate batch: 7/210
Evaluate batch: 8/210
Evaluate batch: 9/210
Evaluate batch: 10/210
Evaluate batch: 11/210
Evaluate batch: 12/210
Evaluate batch: 13/210
Evaluate batch: 14/210
Evaluate batch: 15/210
Evaluate batch: 16/210
Evaluate batch: 17/210
Evaluate batch: 18/210
Evaluate batch: 19/210
Evaluate batch: 20/210
Evaluate batch: 21/210
Evaluate batch: 22/210
Evaluate batch: 23/210
Evaluate batch: 24/210
Evaluate batch: 25/210
Evaluate batch: 26/210
Evaluate batch: 27/210
Evaluate batch: 28/210
Evaluate batch: 29/210
Evaluate batch: 30/210
Evaluate batch: 31/210
Evaluate batch: 32/210
Evaluate batch: 33/210
Evaluate batch: 34/210
Evaluate bat

# INDIVIDUAL SAMPLE TESTING

## Configuration

In [ ]:
####### Sample configuration #######
sample_idx = 242

# 1. Retrieve the original dataset sample.
sample = val_dataset[sample_idx]

# 2. Apply the same collate function as validation.
# A batch containing only one sample.
(   model_input,
    input_lengths,
    target_ids,
    target_lengths
) = dataloader_collate_fn([sample])